# Complete line plot generator

This notebook follows the scatterplot-generator structure, but generates and tests classic line plots. It includes:

- all line-plot parameters from the spreadsheet,
- sampling weights for every parameter option,
- a new `x_axis_month_day_labels` parameter,
- safe color handling for Matplotlib, Plotly, and Altair,
- test filenames that include the tested parameter, code, and value.

In [2]:
from __future__ import annotations

import json
import re
import math
import shutil
from datetime import datetime
from pathlib import Path
from uuid import uuid4

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

try:
    import plotly.graph_objects as go
except Exception:
    go = None

try:
    import altair as alt
except Exception:
    alt = None

In [3]:
PROJECT = Path.cwd().resolve()
OUTPUTS = PROJECT / "outputs"
GENERATED = OUTPUTS / "generated"
GENERATED.mkdir(parents=True, exist_ok=True)

LINE_OUT_ROOT = GENERATED / "lineplots"
LINE_TEST_OUT_ROOT = GENERATED / "line_testing"

## 1. Style schema

In [4]:
STYLE_SPEC = {
    "title_present": {"column": 1, "default": True, "codes": {0: False, 1: True}},
    "title_location": {"column": 2, "default": "center", "codes": {0: "none", 1: "center", 2: "left", 3: "right"}},
    "title_color": {"column": 3, "default": "black", "codes": {0: "black", 1: "orange", 2: "red", 3: "darkgray", 4: None, 5: "green", 6: "lightgray"}},
    "title_size": {"column": 4, "default": "medium", "codes": {0: "medium", 1: "small", 2: "large", 3: None}},
    "subtitle_present": {"column": 5, "default": False, "codes": {0: False, 1: True}},
    "legend_present": {"column": 6, "default": True, "codes": {0: False, 1: True}},
    "legend_title_size": {"column": 7, "default": "none", "codes": {0: "none", 1: "small"}},
    "legend_title_color": {"column": 8, "default": None, "codes": {0: None, 1: "black"}},
    "legend_text_color": {"column": 9, "default": "black", "codes": {0: "black", 1: "same_as_title", 2: None, 3: "darkgray", 4: "lightgray"}},
    "legend_outline": {"column": 10, "default": False, "codes": {0: False, 1: True}},
    "legend_fill": {"column": 11, "default": "none", "codes": {0: "none", 1: "gray_block", 2: "plain", 3: "white_block"}},
    "legend_orientation": {"column": 12, "default": "right", "codes": {0: "none", 1: "left", 2: "right", 3: "top", 4: "bottom", 5: "top_left", 6: "top_right", 7: "inside", 8: "top_horizontal"}},
    "direct_labels": {"column": 13, "default": "none", "codes": {0: "none", 1: "all", 2: "partial"}},
    "label_content": {"column": 14, "default": "none", "codes": {0: "none", 1: "category", 2: "numeric"}},
    "label_color": {"column": 15, "default": "black", "codes": {0: "black", 1: "same_as_title", 2: "black", 3: "bright"}},
    "chart_outline": {"column": 16, "default": "axes_only", "codes": {0: "none", 1: "axes_only", 2: "full", 3: "axes_left"}},
    "gridlines": {"column": 17, "default": "horizontal", "codes": {0: "none", 1: "both", 2: "horizontal", 3: "both_dense", 4: "both_wide"}},
    "gridline_color": {"column": 18, "default": "gray", "codes": {0: None, 1: "gray", 2: "black", 3: "lightgray", 4: "color"}},
    "image_outline": {"column": 19, "default": False, "codes": {0: False, 1: True}},
    "background": {"column": 20, "default": "white", "codes": {0: "transparent", 1: "white", 2: "light_gray", 3: "dark_gray", 4: "light_color"}},
    "axis_text_orientation": {"column": 21, "default": "parallel", "codes": {0: "parallel", 1: "horizontal", 2: "none"}},
    "axis_text_color": {"column": 22, "default": "black", "codes": {0: "black", 1: "same_as_title", 2: "green", 3: None, 4: "lightgray"}},
    "axis_color": {"column": 23, "default": "black", "codes": {0: "black", 1: "darkgray", 2: "lightgray"}},
    "x_scale": {"column": 24, "default": "0_20", "codes": {0: "0_20", 1: "0_50", 2: "years", 3: "months", 4: "0_10", 5: "days"}},
    "x_tick_step": {"column": 25, "default": 5, "codes": {0: 5, 1: 2, 2: "5_years", 3: "1_year", 4: "months", 5: "days", 6: 1}},
    "y_scale": {"column": 26, "default": "0_100", "codes": {0: "0_100", 1: "0_50", 2: "0_1000", 3: "0_10", 4: "0_200", 5: "0_5", 6: "0_100_percent", 7: "2400_2900", 8: "0_2000", 9: "0_30"}},
    "y_tick_step": {"column": 27, "default": 5, "codes": {0: 2, 1: 5, 2: 100, 3: 1, 4: 20, 5: 10, 6: 200, 7: 0.5, 8: 25, 9: 30, 10: 6, 11: 7}},
    "line_orientation": {"column": 28, "default": "ascending", "codes": {0: "descending", 1: "ascending", 2: "up_down", 3: "mixed", 4: "mostly_ascending", 5: "mostly_descending"}},
    "line_structure": {"column": 29, "default": "straight", "codes": {0: "straight", 1: "smooth"}},
    "line_pattern": {"column": 30, "default": "solid", "codes": {0: "solid", 1: "dotted", 2: "dashed", 3: "mixed"}},
    "n_lines": {"column": 31, "default": 1, "codes": {1: 1, 2: 2, 3: 3, 5: 5, 6: 6}},
    "line_start": {"column": 32, "default": "x0", "codes": {0: "x0", 1: "first_x"}},
    "point_shape_mode": {"column": 33, "default": "dots", "codes": {0: "dots", 1: "squares", 2: "by_line", 3: "none"}},
    "point_same_color": {"column": 34, "default": "same", "codes": {0: "different", 1: "same", 2: "shade", 3: "none"}},
    "palette_type": {"column": 35, "default": "bright", "codes": {0: "black", 1: "dark", 2: "bright"}},
    "x_axis_month_day_labels": {"column": 36, "default": "none", "codes": {0: "none", 1: "months_short", 2: "days_short", 3: "days_long"}},
}

## 2. Sampling weights

Every parameter has a `sampling_weights` dictionary. These weights affect normal random generation only. The test generator still creates one plot per option.

In [5]:
CUSTOM_SAMPLING_WEIGHTS = {
    "title_present": {False: 0.08, True: 0.92},
    "title_location": {"none": 0.05, "center": 0.65, "left": 0.20, "right": 0.10},
    "title_color": {"black": 0.70, "orange": 0.06, "red": 0.04, "darkgray": 0.10, None: 0.03, "green": 0.03, "lightgray": 0.04},
    "title_size": {"medium": 0.65, "small": 0.18, "large": 0.14, None: 0.03},
    "subtitle_present": {False: 0.65, True: 0.35},
    "legend_present": {False: 0.20, True: 0.80},
    "legend_title_size": {"none": 0.75, "small": 0.25},
    "legend_title_color": {None: 0.70, "black": 0.30},
    "legend_text_color": {"black": 0.62, "same_as_title": 0.16, None: 0.06, "darkgray": 0.10, "lightgray": 0.06},
    "legend_outline": {False: 0.78, True: 0.22},
    "legend_fill": {"none": 0.70, "gray_block": 0.10, "plain": 0.12, "white_block": 0.08},
    "legend_orientation": {"none": 0.07, "left": 0.08, "right": 0.45, "top": 0.09, "bottom": 0.09, "top_left": 0.06, "top_right": 0.06, "inside": 0.05, "top_horizontal": 0.05},
    "direct_labels": {"none": 0.78, "all": 0.07, "partial": 0.15},
    "label_content": {"none": 0.78, "category": 0.10, "numeric": 0.12},
    "label_color": {"black": 0.58, "same_as_title": 0.18, "bright": 0.08},
    "chart_outline": {"none": 0.12, "axes_only": 0.68, "full": 0.10, "axes_left": 0.10},
    "gridlines": {"none": 0.18, "both": 0.15, "horizontal": 0.52, "both_dense": 0.07, "both_wide": 0.08},
    "gridline_color": {None: 0.08, "gray": 0.55, "black": 0.06, "lightgray": 0.25, "color": 0.06},
    "image_outline": {False: 0.88, True: 0.12},
    "background": {"transparent": 0.02, "white": 0.86, "light_gray": 0.07, "dark_gray": 0.01, "light_color": 0.04},
    "axis_text_orientation": {"parallel": 0.70, "horizontal": 0.22, "none": 0.08},
    "axis_text_color": {"black": 0.68, "same_as_title": 0.14, "green": 0.05, None: 0.04, "lightgray": 0.09},
    "axis_color": {"black": 0.66, "darkgray": 0.22, "lightgray": 0.12},
    "x_scale": {"0_20": 0.42, "0_50": 0.13, "years": 0.16, "months": 0.12, "0_10": 0.10, "days": 0.07},
    "x_tick_step": {5: 0.28, 2: 0.17, "5_years": 0.12, "1_year": 0.09, "months": 0.10, "days": 0.08, 1: 0.16},
    "y_scale": {"0_100": 0.34, "0_50": 0.12, "0_1000": 0.08, "0_10": 0.10, "0_200": 0.09, "0_5": 0.06, "0_100_percent": 0.10, "2400_2900": 0.03, "0_2000": 0.04, "0_30": 0.04},
    "y_tick_step": {2: 0.07, 5: 0.20, 100: 0.10, 1: 0.10, 20: 0.11, 10: 0.16, 200: 0.07, 0.5: 0.04, 25: 0.06, 30: 0.04, 6: 0.025, 7: 0.025},
    "line_orientation": {"descending": 0.16, "ascending": 0.34, "up_down": 0.16, "mixed": 0.10, "mostly_ascending": 0.16, "mostly_descending": 0.08},
    "line_structure": {"straight": 0.72, "smooth": 0.28},
    "line_pattern": {"solid": 0.66, "dotted": 0.11, "dashed": 0.14, "mixed": 0.09},
    "n_lines": {1: 0.38, 2: 0.28, 3: 0.19, 5: 0.09, 6: 0.06},
    "line_start": {"x0": 0.78, "first_x": 0.22},
    "point_shape_mode": {"dots": 0.58, "squares": 0.12, "by_line": 0.16, "none": 0.14},
    "point_same_color": {"different": 0.12, "same": 0.66, "shade": 0.14, "none": 0.08},
    "palette_type": {"black": 0.08, "dark": 0.18, "bright": 0.74},
    "x_axis_month_day_labels": {"none": 0.75, "months_short": 0.12, "days_short": 0.08, "days_long": 0.05},
}


def normalized_weights_for_values(values, weights_dict=None, default_value=None, default_weight=0.70):
    if weights_dict is None:
        if len(values) == 1:
            weights_dict = {values[0]: 1.0}
        else:
            others = [v for v in values if v != default_value]
            other_weight = (1.0 - default_weight) / len(others) if others else 0.0
            weights_dict = {v: (default_weight if v == default_value else other_weight) for v in values}
    probs = np.array([float(weights_dict.get(v, 0.0)) for v in values], dtype=float)
    if probs.sum() <= 0:
        probs = np.ones(len(values), dtype=float)
    probs = probs / probs.sum()
    return {v: float(p) for v, p in zip(values, probs)}


def add_sampling_weights_to_style_spec(style_spec, custom_weights=None, default_weight=0.70):
    custom_weights = custom_weights or {}
    updated = {}
    for param_name, spec in style_spec.items():
        spec = spec.copy()
        values = list(spec["codes"].values())
        spec["sampling_weights"] = normalized_weights_for_values(
            values,
            custom_weights.get(param_name),
            default_value=spec.get("default", values[0]),
            default_weight=default_weight,
        )
        updated[param_name] = spec
    return updated


STYLE_SPEC = add_sampling_weights_to_style_spec(STYLE_SPEC, CUSTOM_SAMPLING_WEIGHTS)


def validate_sampling_weights(style_spec):
    rows = []
    for param_name, spec in style_spec.items():
        values = list(spec["codes"].values())
        weights = spec.get("sampling_weights", {})
        rows.append({
            "parameter": param_name,
            "n_options": len(values),
            "has_all_options": all(v in weights for v in values),
            "weight_sum": sum(weights.get(v, 0.0) for v in values),
        })
    return pd.DataFrame(rows)


validate_sampling_weights(STYLE_SPEC).head()

,parameter,n_options,has_all_options,weight_sum
0,title_present,2,True,1.0
1,title_location,4,True,1.0
2,title_color,7,True,1.0
3,title_size,4,True,1.0
4,subtitle_present,2,True,1.0


## 3. Style helpers and data generation

In [6]:
def new_chart_id(prefix: str = "line") -> str:
    return f"{prefix}_{datetime.utcnow().strftime('%Y%m%dT%H%M%S')}_{uuid4().hex[:8]}"


def safe_slug(value, max_len=140):
    s = str(value)
    s = s.replace("None", "none")
    s = re.sub(r"[^A-Za-z0-9._-]+", "_", s)
    s = s.strip("_")
    return s[:max_len] or "value"


def style_defaults() -> dict:
    return {key: spec["default"] for key, spec in STYLE_SPEC.items()}


def available_options() -> pd.DataFrame:
    rows = []
    for param, spec in STYLE_SPEC.items():
        weights = spec.get("sampling_weights", {})
        for code, value in spec["codes"].items():
            rows.append({"parameter": param, "code": code, "value": value, "sampling_weight": weights.get(value)})
    return pd.DataFrame(rows)


def weighted_choice(rng: np.random.Generator, values: list, weights: dict):
    probs = np.array([weights.get(v, 0.0) for v in values], dtype=float)
    if probs.sum() <= 0:
        probs = np.ones(len(values), dtype=float)
    probs = probs / probs.sum()
    return values[int(rng.choice(len(values), p=probs))]


def sample_style(rng: np.random.Generator, param_counts: dict | None = None, forced_style: dict | None = None) -> dict:
    style = style_defaults()
    for key, spec in STYLE_SPEC.items():
        values = list(spec["codes"].values())
        weights = param_counts.get(key) if param_counts and key in param_counts else spec.get("sampling_weights", {})
        style[key] = weighted_choice(rng, values, weights)
    if forced_style:
        style.update(forced_style)
    return harmonize_style(style)


def harmonize_style(style: dict) -> dict:
    style = dict(style)
    if not style.get("title_present", True):
        style["title_location"] = "none"
        style["title_color"] = None
        style["title_size"] = None
    if style.get("title_location") == "none":
        style["title_present"] = False
    if not style.get("legend_present", True):
        style["legend_orientation"] = "none"
        style["legend_title_size"] = "none"
        style["legend_title_color"] = None
        style["legend_text_color"] = None
        style["legend_outline"] = False
        style["legend_fill"] = "none"
    if style.get("legend_orientation") == "none":
        style["legend_present"] = False
    if style.get("direct_labels") == "none":
        style["label_content"] = "none"
    if style.get("label_content") == "none":
        style["direct_labels"] = "none"
    if style.get("axis_text_orientation") == "none":
        style["axis_text_color"] = None
    if style.get("point_shape_mode") == "none":
        style["point_same_color"] = "none"
    if style.get("point_same_color") == "none":
        style["point_shape_mode"] = "none"
    return style

available_options().head(20)

,parameter,code,value,sampling_weight
0,title_present,0,False,0.08
1,title_present,1,True,0.92
2,title_location,0,none,0.05
3,title_location,1,center,0.65
4,title_location,2,left,0.20
5,title_location,3,right,0.10
6,title_color,0,black,0.70
7,title_color,1,orange,0.06
8,title_color,2,red,0.04
9,title_color,3,darkgray,0.10


In [7]:
def get_special_x_labels(style: dict):
    mode = style.get("x_axis_month_day_labels", "none")
    if mode == "months_short":
        return ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"], "Month"
    if mode == "days_short":
        return ["Mon", "Tues", "Wed", "Thurs", "Fri", "Sat", "Sun"], "Day"
    if mode == "days_long":
        return ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"], "Day"
    return None, None


def get_x_values(style: dict):
    special_labels, special_axis_label = get_special_x_labels(style)
    if special_labels is not None:
        positions = np.arange(len(special_labels))
        return positions, special_axis_label, special_labels

    scale = style.get("x_scale", "0_20")
    if scale == "0_50":
        x = np.arange(0, 51, 5)
        return x, "Measurement step", [str(v) for v in x]
    if scale == "0_10":
        x = np.arange(0, 11, 1)
        return x, "Measurement step", [str(v) for v in x]
    if scale == "years":
        x = np.arange(2000, 2026, 2)
        return x, "Year", [str(v) for v in x]
    if scale == "months":
        labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
        x = np.arange(1, 13)
        return x, "Month", labels
    if scale == "days":
        labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
        x = np.arange(1, 8)
        return x, "Day", labels
    x = np.arange(0, 21, 2)
    return x, "Measurement step", [str(v) for v in x]


def get_y_limits(style: dict):
    return {
        "0_50": (0, 50),
        "0_100": (0, 100),
        "0_1000": (0, 1000),
        "0_10": (0, 10),
        "0_200": (0, 200),
        "0_5": (0, 5),
        "0_100_percent": (0, 100),
        "2400_2900": (2400, 2900),
        "0_2000": (0, 2000),
        "0_30": (0, 30),
    }.get(style.get("y_scale", "0_100"), (0, 100))


def make_base_line(x: np.ndarray, y_min: float, y_max: float, orientation: str, rng: np.random.Generator) -> np.ndarray:
    n = len(x)
    span = y_max - y_min
    low = y_min + span * 0.18
    high = y_min + span * 0.82
    if orientation == "ascending":
        y = np.linspace(low, high, n)
    elif orientation == "descending":
        y = np.linspace(high, low, n)
    elif orientation == "mostly_ascending":
        y = np.linspace(low, high, n) + rng.normal(0, span * 0.07, n)
    elif orientation == "mostly_descending":
        y = np.linspace(high, low, n) + rng.normal(0, span * 0.07, n)
    elif orientation == "mixed":
        trend = rng.choice(["ascending", "descending", "up_down"])
        return make_base_line(x, y_min, y_max, trend, rng)
    else:
        y = y_min + span * (0.5 + 0.25 * np.sin(np.linspace(0, 2.5 * np.pi, n)))
    y += rng.normal(0, span * 0.035, n)
    return np.clip(y, y_min, y_max)


def sample_line_data(rng: np.random.Generator, style: dict) -> tuple[pd.DataFrame, dict]:
    x, x_label, x_tick_labels = get_x_values(style)
    y_min, y_max = get_y_limits(style)
    n_lines = int(style.get("n_lines", 1))
    if style.get("line_start") == "first_x" and len(x) > 1 and x[0] == 0:
        x = x[1:]
        x_tick_labels = x_tick_labels[1:]
    rows = []
    for i in range(n_lines):
        y = make_base_line(x, y_min, y_max, style.get("line_orientation", "ascending"), rng)
        offset = (i - (n_lines - 1) / 2) * (y_max - y_min) * 0.06
        y = np.clip(y + offset, y_min, y_max)
        for j, (xv, yv) in enumerate(zip(x, y)):
            rows.append({"x": xv, "x_label": x_tick_labels[j], "y": float(yv), "series": f"Line {i + 1}"})
    df = pd.DataFrame(rows)
    context = {
        "x_label": x_label,
        "y_label": "Value (%)" if style.get("y_scale") == "0_100_percent" else "Value",
        "title": "Line plot example",
        "subtitle": "Synthetic data generated from selected line-chart parameters",
        "x_tick_positions": list(map(float, x)),
        "x_tick_labels": list(x_tick_labels),
        "x_axis_month_day_labels": style.get("x_axis_month_day_labels", "none"),
    }
    return df, context

## 4. Shared plotting helpers

In [8]:
BRIGHT_COLORS = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#17becf"]
DARK_COLORS = ["#111111", "#3a3a3a", "#555555", "#23415a", "#4a2c5f", "#31533b"]
BLACK_COLORS = ["#000000"] * 6
MARKERS = ["o", "s", "^", "D", "P", "X"]
PLOTLY_MARKERS = ["circle", "square", "triangle-up", "diamond", "cross", "x"]
LINE_DASHES = {"solid": "solid", "dotted": "dot", "dashed": "dash"}
MPL_LINESTYLES = {"solid": "-", "dotted": ":", "dashed": "--"}


def color_list(style: dict, n: int) -> list[str]:
    palette = style.get("palette_type", "bright")
    base = BLACK_COLORS if palette == "black" else DARK_COLORS if palette == "dark" else BRIGHT_COLORS
    return [base[i % len(base)] for i in range(n)]


def color_value(name, style=None, line_color=None, fallback=None):
    if name is None or name in [False, "none", "None", ""]:
        return fallback
    base_colors = {
        "black": "#000000",
        "orange": "#ff7f0e",
        "red": "#d62728",
        "darkgray": "#555555",
        "dark_gray": "#555555",
        "lightgray": "#c7c7c7",
        "light_gray": "#e6e6e6",
        "very_light_gray": "#f5f5f5",
        "green": "#2ca02c",
        "gray": "#8c8c8c",
        "grey": "#8c8c8c",
        "white": "#ffffff",
        "color": "#1f77b4",
        "bright": "#d62728",
        "light_color": "#edf4ff",
        "transparent": "rgba(0,0,0,0)",
    }
    if name == "same_as_title":
        return color_value(style.get("title_color") if style else "black", style=style, fallback=fallback or "#000000")
    if name in ["same_as_line", "same_as_series"]:
        return line_color or fallback
    if name in base_colors:
        return base_colors[name]
    if isinstance(name, str) and (name.startswith("#") or name.startswith("C") or name.startswith("rgb")):
        return name
    return fallback


def background_color(style: dict):
    return {
        "transparent": "none",
        "white": "#ffffff",
        "light_gray": "#f2f2f2",
        "dark_gray": "#4d4d4d",
        "light_color": "#edf4ff",
    }.get(style.get("background"), "#ffffff")


def title_font_size(style: dict):
    return {"small": 11, "medium": 14, "large": 18}.get(style.get("title_size"), 14)


def legend_location(style: dict):
    return {
        "left": ("center left", (-0.22, 0.5)),
        "right": ("center left", (1.02, 0.5)),
        "top": ("lower center", (0.5, 1.04)),
        "top_horizontal": ("lower center", (0.5, 1.04)),
        "bottom": ("upper center", (0.5, -0.12)),
        "top_left": ("lower left", (0, 1.02)),
        "top_right": ("lower right", (1, 1.02)),
        "inside": ("best", None),
    }.get(style.get("legend_orientation"), ("best", None))


def line_pattern_for(style: dict, idx: int):
    pattern = style.get("line_pattern", "solid")
    if pattern == "mixed":
        return ["solid", "dotted", "dashed"][idx % 3]
    return pattern


def marker_for(style: dict, idx: int):
    mode = style.get("point_shape_mode", "dots")
    if mode == "none":
        return None
    if mode == "squares":
        return "s"
    if mode == "by_line":
        return MARKERS[idx % len(MARKERS)]
    return "o"


def point_color(line_color: str, style: dict, idx: int):
    mode = style.get("point_same_color", "same")
    if mode == "none":
        return None
    if mode == "different":
        return BRIGHT_COLORS[(idx + 2) % len(BRIGHT_COLORS)]
    if mode == "shade":
        return "#999999"
    return line_color

## 5. Matplotlib renderer

In [9]:
def apply_mpl_axes_style(ax, fig, style: dict, context: dict):
    bg = background_color(style)
    if bg != "none":
        fig.patch.set_facecolor(bg)
        ax.set_facecolor(bg)
    else:
        fig.patch.set_alpha(0)
        ax.set_facecolor("none")

    if style.get("title_present") and style.get("title_location") != "none":
        loc = {"center": "center", "left": "left", "right": "right"}.get(style.get("title_location"), "center")
        title = context["title"]
        if style.get("subtitle_present"):
            title += "\\n" + context["subtitle"]
        title_c = color_value(style.get("title_color"), style=style, fallback="#000000")
        ax.set_title(title, loc=loc, fontsize=title_font_size(style), color=title_c)

    if context.get("x_tick_positions") is not None and context.get("x_tick_labels") is not None:
        ax.set_xticks(context["x_tick_positions"])
        ax.set_xticklabels(context["x_tick_labels"])

    if style.get("axis_text_orientation") != "none":
        axis_color_text = color_value(style.get("axis_text_color"), style=style, fallback="#000000")
        ax.set_xlabel(context["x_label"], color=axis_color_text)
        ax.set_ylabel(context["y_label"], color=axis_color_text)
        ax.tick_params(axis="both", colors=axis_color_text)
    else:
        ax.set_xlabel("")
        ax.set_ylabel("")
        ax.set_xticklabels([])
        ax.set_yticklabels([])

    axis_c = color_value(style.get("axis_color"), style=style, fallback="#000000")
    for spine in ax.spines.values():
        spine.set_color(axis_c)
    if style.get("chart_outline") == "none":
        for spine in ax.spines.values():
            spine.set_visible(False)
    elif style.get("chart_outline") == "axes_only":
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
    elif style.get("chart_outline") == "axes_left":
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.yaxis.tick_left()
    else:
        for spine in ax.spines.values():
            spine.set_visible(True)

    grid = style.get("gridlines")
    if grid != "none":
        axis = "both" if grid in ["both", "both_dense", "both_wide"] else "y"
        grid_color = color_value(style.get("gridline_color"), style=style, fallback="#cccccc")
        width = 0.4 if grid == "both_dense" else 0.8 if grid == "both_wide" else 0.6
        ax.grid(True, axis=axis, color=grid_color, linewidth=width, alpha=0.75)
    else:
        ax.grid(False)

    if style.get("axis_text_orientation") == "parallel":
        plt.setp(ax.get_xticklabels(), rotation=0)
        plt.setp(ax.get_yticklabels(), rotation=90)
    elif style.get("axis_text_orientation") == "horizontal":
        plt.setp(ax.get_xticklabels(), rotation=0)
        plt.setp(ax.get_yticklabels(), rotation=0)

    y_min, y_max = get_y_limits(style)
    ax.set_ylim(y_min, y_max)
    step = style.get("y_tick_step")
    if isinstance(step, (int, float)) and step > 0:
        ticks = np.arange(y_min, y_max + step, step)
        if len(ticks) <= 25:
            ax.set_yticks(ticks)
    if style.get("y_scale") == "0_100_percent":
        ax.set_yticklabels([f"{int(t)}%" for t in ax.get_yticks()])

    if style.get("image_outline"):
        fig.patch.set_edgecolor("black")
        fig.patch.set_linewidth(2)


def add_mpl_legend(ax, style: dict):
    if not style.get("legend_present") or style.get("legend_orientation") == "none":
        return
    loc, bbox = legend_location(style)
    frameon = bool(style.get("legend_outline") or style.get("legend_fill") in ["gray_block", "white_block"])
    kwargs = {"loc": loc, "frameon": frameon}
    if bbox is not None:
        kwargs["bbox_to_anchor"] = bbox
    if style.get("legend_orientation") in ["top", "bottom", "top_horizontal"]:
        kwargs["ncol"] = 3
    title = "Series" if style.get("legend_title_size") != "none" else None
    leg = ax.legend(title=title, **kwargs)
    if leg:
        if style.get("legend_fill") == "gray_block":
            leg.get_frame().set_facecolor("#eeeeee")
        elif style.get("legend_fill") == "white_block":
            leg.get_frame().set_facecolor("#ffffff")
        txt_color = color_value(style.get("legend_text_color"), style=style, fallback="#000000")
        for text in leg.get_texts():
            text.set_color(txt_color)
        if leg.get_title():
            leg.get_title().set_color(color_value(style.get("legend_title_color"), style=style, fallback=txt_color))


def render_line_matplotlib(df: pd.DataFrame, context: dict, style: dict):
    series_names = list(df["series"].unique())
    colors = color_list(style, len(series_names))
    fig, ax = plt.subplots(figsize=(7.2, 4.6), dpi=140)
    for idx, series in enumerate(series_names):
        sub = df[df["series"] == series].sort_values("x")
        lc = colors[idx]
        marker = marker_for(style, idx)
        pc = point_color(lc, style, idx)
        pattern = line_pattern_for(style, idx)
        if style.get("line_structure") == "smooth" and len(sub) >= 4:
            xs = np.linspace(sub["x"].min(), sub["x"].max(), 160)
            ys = np.interp(xs, sub["x"], sub["y"])
            ax.plot(xs, ys, linestyle=MPL_LINESTYLES.get(pattern, "-"), color=lc, linewidth=2, label=series)
            if marker is not None:
                ax.scatter(sub["x"], sub["y"], marker=marker, color=pc, s=28, zorder=3)
        else:
            ax.plot(sub["x"], sub["y"], linestyle=MPL_LINESTYLES.get(pattern, "-"), color=lc, marker=marker, markerfacecolor=pc, markeredgecolor=pc, linewidth=2, label=series)

        if style.get("direct_labels") != "none" and style.get("label_content") != "none":
            points = sub if style.get("direct_labels") == "all" else sub.iloc[::max(1, len(sub)//4)]
            for _, row in points.iterrows():
                text = series if style.get("label_content") == "category" else f"{row['y']:.1f}"
                label_c = color_value(style.get("label_color"), style=style, line_color=lc, fallback=lc)
                ax.text(row["x"], row["y"], text, fontsize=7, color=label_c, ha="left", va="bottom")

    apply_mpl_axes_style(ax, fig, style, context)
    add_mpl_legend(ax, style)
    fig.tight_layout()
    return fig

## 6. Plotly and Altair renderers

In [30]:
def render_line_plotly(df: pd.DataFrame, context: dict, style: dict):
    if go is None:
        raise ImportError("plotly is not installed")
    series_names = list(df["series"].unique())
    colors = color_list(style, len(series_names))
    fig = go.Figure()
    for idx, series in enumerate(series_names):
        sub = df[df["series"] == series].sort_values("x")
        marker_symbol = None if marker_for(style, idx) is None else PLOTLY_MARKERS[idx % len(PLOTLY_MARKERS)] if style.get("point_shape_mode") == "by_line" else ("square" if style.get("point_shape_mode") == "squares" else "circle")
        mode = "lines" if marker_symbol is None else "lines+markers"
        text = None
        if style.get("direct_labels") != "none" and style.get("label_content") != "none":
            text = [series if style.get("label_content") == "category" else f"{v:.1f}" for v in sub["y"]]
            mode += "+text"
        fig.add_trace(go.Scatter(
            x=sub["x"], y=sub["y"], name=series, mode=mode,
            line={"color": colors[idx], "dash": LINE_DASHES.get(line_pattern_for(style, idx), "solid"), "shape": "spline" if style.get("line_structure") == "smooth" else "linear"},
            marker={"symbol": marker_symbol or "circle", "color": point_color(colors[idx], style, idx) or colors[idx]},
            text=text, textposition="top center",
        ))
    bg = "rgba(0,0,0,0)" if background_color(style) == "none" else background_color(style)
    title = None
    if style.get("title_present") and style.get("title_location") != "none":
        title = context["title"] + (f"<br><sup>{context['subtitle']}</sup>" if style.get("subtitle_present") else "")
    legend_orientation = style.get("legend_orientation")
    fig.update_layout(
        title={"text": title, "x": {"left": 0.02, "center": 0.5, "right": 0.98}.get(style.get("title_location"), 0.5), "font": {"size": title_font_size(style), "color": color_value(style.get("title_color"), style=style, fallback="#000000")}},
        xaxis_title=context["x_label"] if style.get("axis_text_orientation") != "none" else None,
        yaxis_title=context["y_label"] if style.get("axis_text_orientation") != "none" else None,
        plot_bgcolor=bg, paper_bgcolor=bg, width=760, height=480,
        showlegend=bool(style.get("legend_present") and style.get("legend_orientation") != "none"),
        legend={"orientation": "h" if legend_orientation in ["top", "bottom", "top_horizontal"] else "v"},
    )
    y_min, y_max = get_y_limits(style)
    fig.update_yaxes(range=[y_min, y_max], dtick=style.get("y_tick_step") if isinstance(style.get("y_tick_step"), (int, float)) else None)
    if context.get("x_tick_positions") is not None and context.get("x_tick_labels") is not None:
        fig.update_xaxes(tickmode="array", tickvals=context["x_tick_positions"], ticktext=context["x_tick_labels"])
    grid = style.get("gridlines")
    grid_color = color_value(style.get("gridline_color"), style=style, fallback="#cccccc")
    axis_text_c = color_value(style.get("axis_text_color"), style=style, fallback="#000000")
    fig.update_xaxes(showgrid=grid in ["both", "both_dense", "both_wide"], gridcolor=grid_color, tickfont={"color": axis_text_c}, title_font={"color": axis_text_c})
    fig.update_yaxes(showgrid=grid != "none", gridcolor=grid_color, tickfont={"color": axis_text_c}, title_font={"color": axis_text_c})
    return fig

def render_line_altair(df: pd.DataFrame, context: dict, style: dict):
    if alt is None:
        raise ImportError("altair is not installed")

    n = df["series"].nunique()
    colors = color_list(style, n)
    y_min, y_max = get_y_limits(style)

    use_named_x = (
        context.get("x_axis_month_day_labels", "none") != "none"
        or style.get("x_scale") in ["months", "days"]
    )

    if use_named_x:
        x_enc = alt.X(
            "x_label:N",
            sort=context.get("x_tick_labels"),
            title=context["x_label"] if style.get("axis_text_orientation") != "none" else None,
        )
        label_x_enc = alt.X(
            "x_label:N",
            sort=context.get("x_tick_labels"),
            title=None,
        )
    else:
        x_enc = alt.X(
            "x:Q",
            title=context["x_label"] if style.get("axis_text_orientation") != "none" else None,
        )
        label_x_enc = alt.X("x:Q", title=None)

    legend_obj = (
        None
        if not style.get("legend_present") or style.get("legend_orientation") == "none"
        else alt.Legend(
            title="Series" if style.get("legend_title_size") != "none" else None
        )
    )

    base = alt.Chart(df).encode(
        x=x_enc,
        y=alt.Y(
            "y:Q",
            title=context["y_label"] if style.get("axis_text_orientation") != "none" else None,
            scale=alt.Scale(domain=[y_min, y_max]),
        ),
        color=alt.Color(
            "series:N",
            scale=alt.Scale(range=colors),
            legend=legend_obj,
        ),
    )

    interpolate_mode = (
        "monotone"
        if style.get("line_structure") == "smooth"
        else "linear"
    )

    if style.get("line_pattern") == "mixed":
        line = base.mark_line(interpolate=interpolate_mode).encode(
            strokeDash=alt.StrokeDash("series:N")
        )
    else:
        line = base.mark_line(interpolate=interpolate_mode)

    layers = [line]

    if style.get("point_shape_mode") != "none":
        if style.get("point_shape_mode") == "by_line":
            shape = alt.Shape("series:N")
        else:
            shape = alt.value(
                "square"
                if style.get("point_shape_mode") == "squares"
                else "circle"
            )

        layers.append(
            base.mark_point(size=65, filled=True).encode(
                shape=shape
            )
        )

    if style.get("direct_labels") != "none" and style.get("label_content") != "none":

        if style.get("direct_labels") == "all":
            label_df = df.copy()
        else:
            selected_parts = []

            for series_name in df["series"].unique():
                series_df = df[df["series"] == series_name].copy()
                step = max(1, len(series_df) // 4)
                selected_parts.append(series_df.iloc[::step].copy())

            if selected_parts:
                label_df = pd.concat(selected_parts, ignore_index=True)
            else:
                label_df = df.iloc[0:0].copy()

        if style.get("label_content") == "category":
            label_df["label"] = label_df["series"].astype(str)
        else:
            label_df["label"] = label_df["y"].round(1).astype(str)

        layers.append(
            alt.Chart(label_df)
            .mark_text(
                align="left",
                dx=4,
                dy=-4,
                fontSize=9,
                color=color_value(
                    style.get("label_color"),
                    style=style,
                    fallback="#000000",
                ),
            )
            .encode(
                x=label_x_enc,
                y=alt.Y("y:Q", title=None),
                text=alt.Text("label:N"),
            )
        )

    # This line was missing in your current code
    chart = alt.layer(*layers).properties(
        width=640,
        height=390,
    )

    if style.get("title_present") and style.get("title_location") != "none":
        anchor = {
            "left": "start",
            "center": "middle",
            "right": "end",
        }.get(style.get("title_location"), "middle")

        title = (
            [context["title"], context["subtitle"]]
            if style.get("subtitle_present")
            else context["title"]
        )

        chart = chart.properties(
            title=alt.TitleParams(
                text=title,
                anchor=anchor,
                color=color_value(
                    style.get("title_color"),
                    style=style,
                    fallback="#000000",
                ),
                fontSize=title_font_size(style),
            )
        )

    return chart.configure_view(
        stroke=None if style.get("chart_outline") == "none" else "black"
    ).configure_axis(
        grid=style.get("gridlines") != "none",
        gridColor=color_value(
            style.get("gridline_color"),
            style=style,
            fallback="#cccccc",
        ),
        labelColor=color_value(
            style.get("axis_text_color"),
            style=style,
            fallback="#000000",
        ),
        titleColor=color_value(
            style.get("axis_text_color"),
            style=style,
            fallback="#000000",
        ),
    )

## 7. Saving and generation

In [31]:
def ensure_output_dirs(out_root: Path):
    for sub in ["images/matplotlib", "images/plotly", "images/altair", "tables", "metadata"]:
        (out_root / sub).mkdir(parents=True, exist_ok=True)


def save_matplotlib_png(fig, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)


def save_plotly(fig, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        fig.write_image(str(path))
    except Exception:
        path = path.with_suffix(".html")
        fig.write_html(str(path))
    return path


def save_altair(chart, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        chart.save(str(path))
    except Exception:
        path = path.with_suffix(".html")
        chart.save(str(path))
    return path


def generate_line(out_root: Path, library: str = "matplotlib", rng_seed: int = 2026, param_counts: dict | None = None, forced_style: dict | None = None, filename_override: str | None = None) -> dict:
    ensure_output_dirs(out_root)
    rng = np.random.default_rng(rng_seed)
    style = sample_style(rng, param_counts=param_counts, forced_style=forced_style)
    df, context = sample_line_data(rng, style)
    chart_id = safe_slug(filename_override) if filename_override else new_chart_id("line")

    if library == "matplotlib":
        chart_path = out_root / "images" / library / f"{chart_id}.png"
        fig = render_line_matplotlib(df, context, style)
        save_matplotlib_png(fig, chart_path)
    elif library == "plotly":
        chart_path = out_root / "images" / library / f"{chart_id}.png"
        fig = render_line_plotly(df, context, style)
        chart_path = save_plotly(fig, chart_path)
    elif library == "altair":
        chart_path = out_root / "images" / library / f"{chart_id}.png"
        chart = render_line_altair(df, context, style)
        chart_path = save_altair(chart, chart_path)
    else:
        raise ValueError(f"Unknown library: {library}")

    table_path = out_root / "tables" / f"{chart_id}.csv"
    meta_path = out_root / "metadata" / f"{chart_id}.json"
    df.to_csv(table_path, index=False)
    meta = {
        "chart_id": chart_id,
        "library": library,
        "image_path": str(chart_path),
        "table_path": str(table_path),
        "style": style,
        "context": context,
        "rng_seed": rng_seed,
    }
    meta_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")
    return meta

## 8. Example generation

In [32]:
meta = generate_line(LINE_OUT_ROOT, library="matplotlib", rng_seed=2026)
meta

{'chart_id': 'line_20260426T110551_dd53d389',
 'library': 'matplotlib',
 'image_path': 'C:\\Users\\Michelle\\I2R\\notebooks\\outputs\\generated\\lineplots\\images\\matplotlib\\line_20260426T110551_dd53d389.png',
 'table_path': 'C:\\Users\\Michelle\\I2R\\notebooks\\outputs\\generated\\lineplots\\tables\\line_20260426T110551_dd53d389.csv',
 'style': {'title_present': True,
  'title_location': 'center',
  'title_color': 'black',
  'title_size': 'medium',
  'subtitle_present': False,
  'legend_present': True,
  'legend_title_size': 'small',
  'legend_title_color': None,
  'legend_text_color': 'same_as_title',
  'legend_outline': False,
  'legend_fill': 'white_block',
  'legend_orientation': 'inside',
  'direct_labels': 'none',
  'label_content': 'none',
  'label_color': 'same_as_title',
  'chart_outline': 'full',
  'gridlines': 'horizontal',
  'gridline_color': 'gray',
  'image_outline': False,
  'background': 'white',
  'axis_text_orientation': 'parallel',
  'axis_text_color': 'black',
  

## 9. Test one image per individual parameter option

The Altair-only test and the all-libraries test both save into `LINE_TEST_OUT_ROOT`, i.e. `outputs/generated/line_testing/`.
Whenever you rerun either test cell with `clean_first=True`, the complete old `line_testing` folder is deleted first and then recreated.
All test images therefore always end up in `line_testing/images/<library>/`.


In [33]:
def clean_line_testing_folder(out_root: Path = LINE_TEST_OUT_ROOT):
    """Delete the complete old line_testing folder and recreate the folder structure."""
    if out_root.exists():
        shutil.rmtree(out_root)
    ensure_output_dirs(out_root)


def _parameter_test_summary_rows(metas: list[dict]) -> pd.DataFrame:
    return pd.DataFrame([
        {
            "chart_id": m.get("chart_id"),
            "library": m.get("library"),
            "tested_parameter": m.get("tested_parameter"),
            "tested_code": m.get("tested_code"),
            "tested_value": m.get("tested_value"),
            "image_path": m.get("image_path"),
            "table_path": m.get("table_path"),
            "rng_seed": m.get("rng_seed"),
        }
        for m in metas
    ])


def _execute_parameter_tests(
    out_root: Path,
    libraries: tuple[str, ...],
    summary_filename: str,
    error_filename: str,
    clean_first: bool = True,
    continue_on_error: bool = False,
    seed_start: int = 50_000,
) -> tuple[list[dict], list[dict]]:
    """
    Core helper used by both the Altair-only test and the all-libraries test.

    If clean_first=True, the complete old line_testing folder is deleted before regeneration.
    All images are saved in out_root / 'images' / <library>.
    """
    if clean_first:
        clean_line_testing_folder(out_root)
    else:
        ensure_output_dirs(out_root)

    metas: list[dict] = []
    errors: list[dict] = []
    seed = seed_start

    for library in libraries:
        for param_name, spec in STYLE_SPEC.items():
            for code_value, decoded_value in spec["codes"].items():
                forced_style = {param_name: decoded_value}
                filename_override = f"{library}__{param_name}__{code_value}__{decoded_value}"

                try:
                    meta = generate_line(
                        out_root=out_root,
                        library=library,
                        rng_seed=seed,
                        forced_style=forced_style,
                        filename_override=filename_override,
                    )
                    meta["test_type"] = "direct_parameter_option"
                    meta["tested_parameter"] = param_name
                    meta["tested_code"] = code_value
                    meta["tested_value"] = decoded_value
                    metas.append(meta)
                except Exception as e:
                    error_record = {
                        "library": library,
                        "tested_parameter": param_name,
                        "tested_code": code_value,
                        "tested_value": decoded_value,
                        "rng_seed": seed,
                        "filename_override": safe_slug(filename_override),
                        "error_type": type(e).__name__,
                        "error_message": str(e),
                    }
                    errors.append(error_record)
                    if not continue_on_error:
                        error_df = pd.DataFrame(errors)
                        if not error_df.empty:
                            error_df.to_csv(out_root / "metadata" / error_filename, index=False)
                        raise
                finally:
                    seed += 1

    summary_df = _parameter_test_summary_rows(metas)
    summary_df.to_csv(out_root / "metadata" / summary_filename, index=False)

    error_df = pd.DataFrame(errors)
    error_df.to_csv(out_root / "metadata" / error_filename, index=False)
    return metas, errors


def run_altair_parameter_tests(
    out_root: Path = LINE_TEST_OUT_ROOT,
    clean_first: bool = True,
    continue_on_error: bool = True,
) -> tuple[list[dict], list[dict]]:
    """
    Generate one Altair chart for every parameter option.

    Saved outputs:
    - images:   out_root / 'images' / 'altair'
    - tables:   out_root / 'tables'
    - metadata: out_root / 'metadata'
    """
    return _execute_parameter_tests(
        out_root=out_root,
        libraries=("altair",),
        summary_filename="altair_parameter_test_summary.csv",
        error_filename="altair_parameter_test_errors.csv",
        clean_first=clean_first,
        continue_on_error=continue_on_error,
        seed_start=80_000,
    )


def run_all_parameter_tests(
    out_root: Path = LINE_TEST_OUT_ROOT,
    clean_first: bool = True,
    continue_on_error: bool = False,
    libraries: tuple[str, ...] = ("matplotlib", "plotly", "altair"),
) -> tuple[list[dict], list[dict]]:
    """
    Generate one test chart for every parameter option for all requested libraries.

    Saved outputs:
    - images:   out_root / 'images' / 'matplotlib' | 'plotly' | 'altair'
    - tables:   out_root / 'tables'
    - metadata: out_root / 'metadata'

    If clean_first=True, rerunning this function always deletes the old line_testing
    folder first, so the folder contents are always fresh.
    """
    return _execute_parameter_tests(
        out_root=out_root,
        libraries=libraries,
        summary_filename="line_parameter_test_summary.csv",
        error_filename="line_parameter_test_errors.csv",
        clean_first=clean_first,
        continue_on_error=continue_on_error,
        seed_start=50_000,
    )


### 9A. Altair-only parameter test

This cell deletes the old `line_testing` folder, reruns the Altair-only parameter test suite, and saves the outputs in `line_testing/images/altair/`.


In [34]:
altair_test_metas, altair_test_errors = run_altair_parameter_tests(
    out_root=LINE_TEST_OUT_ROOT,
    clean_first=True,
    continue_on_error=True,
)

print(f"Altair-only successes: {len(altair_test_metas)}")
print(f"Altair-only errors: {len(altair_test_errors)}")
print(f"Altair image folder: {LINE_TEST_OUT_ROOT / 'images' / 'altair'}")

pd.DataFrame(altair_test_metas).head(), pd.DataFrame(altair_test_errors).head()


Altair-only successes: 156
Altair-only errors: 0
Altair image folder: C:\Users\Michelle\I2R\notebooks\outputs\generated\line_testing\images\altair


(                            chart_id library  \
 0    altair__title_present__0__False  altair   
 1     altair__title_present__1__True  altair   
 2    altair__title_location__0__none  altair   
 3  altair__title_location__1__center  altair   
 4    altair__title_location__2__left  altair   
 
                                           image_path  \
 0  C:\Users\Michelle\I2R\notebooks\outputs\genera...   
 1  C:\Users\Michelle\I2R\notebooks\outputs\genera...   
 2  C:\Users\Michelle\I2R\notebooks\outputs\genera...   
 3  C:\Users\Michelle\I2R\notebooks\outputs\genera...   
 4  C:\Users\Michelle\I2R\notebooks\outputs\genera...   
 
                                           table_path  \
 0  C:\Users\Michelle\I2R\notebooks\outputs\genera...   
 1  C:\Users\Michelle\I2R\notebooks\outputs\genera...   
 2  C:\Users\Michelle\I2R\notebooks\outputs\genera...   
 3  C:\Users\Michelle\I2R\notebooks\outputs\genera...   
 4  C:\Users\Michelle\I2R\notebooks\outputs\genera...   
 
                

In [35]:
altair_test_metas, altair_test_errors = run_altair_parameter_tests(
    out_root=LINE_TEST_OUT_ROOT,
    clean_first=True,
    continue_on_error=False,
)

# print(f"Altair-only successes: {len(altair_test_metas)}")
# print(f"Altair-only errors: {len(altair_test_errors)}")
# print(f"Altair image folder: {LINE_TEST_OUT_ROOT / 'images' / 'altair'}")

# pd.DataFrame(altair_test_metas).head(), pd.DataFrame(altair_test_errors).head()


In [17]:
altair_error_df = pd.DataFrame(altair_test_errors)

altair_error_df[
    [
        "library",
        "tested_parameter",
        "tested_code",
        "tested_value",
        "error_type",
        "error_message",
        "rng_seed",
        "filename_override",
    ]
]

,library,tested_parameter,tested_code,tested_value,error_type,error_message,rng_seed,filename_override
0,altair,label_content,1,category,KeyError,'series',80049,altair__label_content__1__category
1,altair,label_color,3,bright,KeyError,'series',80054,altair__label_color__3__bright
2,altair,x_tick_step,2,5_years,KeyError,'series',80095,altair__x_tick_step__2__5_years
3,altair,line_structure,0,straight,KeyError,'series',80128,altair__line_structure__0__straight


### 9B. All-libraries parameter test

This cell also deletes the old `line_testing` folder first, then regenerates the full test suite for Matplotlib, Plotly, and Altair.
All outputs are saved under `line_testing/`, specifically `line_testing/images/matplotlib/`, `line_testing/images/plotly/`, and `line_testing/images/altair/`.


In [36]:
direct_metas, direct_errors = run_all_parameter_tests(
    out_root=LINE_TEST_OUT_ROOT,
    clean_first=True,
    continue_on_error=False,
)

print(f"All-library successes: {len(direct_metas)}")
print(f"All-library errors: {len(direct_errors)}")
print(f"Matplotlib image folder: {LINE_TEST_OUT_ROOT / 'images' / 'matplotlib'}")
print(f"Plotly image folder: {LINE_TEST_OUT_ROOT / 'images' / 'plotly'}")
print(f"Altair image folder: {LINE_TEST_OUT_ROOT / 'images' / 'altair'}")

len(direct_metas), pd.DataFrame(direct_metas).head()


C:\Users\Michelle\AppData\Local\Temp\ipykernel_24912\3355121882.py:74: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels([f"{int(t)}%" for t in ax.get_yticks()])
Resorting to unclean kill browser.


All-library successes: 468
All-library errors: 0
Matplotlib image folder: C:\Users\Michelle\I2R\notebooks\outputs\generated\line_testing\images\matplotlib
Plotly image folder: C:\Users\Michelle\I2R\notebooks\outputs\generated\line_testing\images\plotly
Altair image folder: C:\Users\Michelle\I2R\notebooks\outputs\generated\line_testing\images\altair


(468,
                                 chart_id     library  \
 0    matplotlib__title_present__0__False  matplotlib   
 1     matplotlib__title_present__1__True  matplotlib   
 2    matplotlib__title_location__0__none  matplotlib   
 3  matplotlib__title_location__1__center  matplotlib   
 4    matplotlib__title_location__2__left  matplotlib   
 
                                           image_path  \
 0  C:\Users\Michelle\I2R\notebooks\outputs\genera...   
 1  C:\Users\Michelle\I2R\notebooks\outputs\genera...   
 2  C:\Users\Michelle\I2R\notebooks\outputs\genera...   
 3  C:\Users\Michelle\I2R\notebooks\outputs\genera...   
 4  C:\Users\Michelle\I2R\notebooks\outputs\genera...   
 
                                           table_path  \
 0  C:\Users\Michelle\I2R\notebooks\outputs\genera...   
 1  C:\Users\Michelle\I2R\notebooks\outputs\genera...   
 2  C:\Users\Michelle\I2R\notebooks\outputs\genera...   
 3  C:\Users\Michelle\I2R\notebooks\outputs\genera...   
 4  C:\Users\Michelle

In [ ]:
def summarize_test_coverage(metas: list[dict]) -> pd.DataFrame:
    rows = []
    meta_df = pd.DataFrame(metas)
    if meta_df.empty:
        return pd.DataFrame(columns=["parameter", "library", "expected_options", "generated_options", "complete"])
    for param, spec in STYLE_SPEC.items():
        expected = len(spec["codes"])
        for library in sorted(meta_df["library"].dropna().unique()):
            found = meta_df[(meta_df["tested_parameter"] == param) & (meta_df["library"] == library)]["tested_code"].nunique()
            rows.append({"parameter": param, "library": library, "expected_options": expected, "generated_options": found, "complete": found == expected})
    return pd.DataFrame(rows)

coverage = summarize_test_coverage(direct_metas)
coverage.to_csv(LINE_TEST_OUT_ROOT / "metadata" / "line_parameter_test_coverage.csv", index=False)
coverage.head(20), coverage[~coverage["complete"]], pd.DataFrame(direct_errors).head()
